# 3D reconstruction benchmark

This notebook invokes core quality assessment, planning, execution and independent validation. It contains orchestration and visualizations, not alternative production algorithms.

**Flow:** inputs -> core RANSAC and feasibility -> optional planned execution -> backend outputs -> common QC and spatial residuals. Notebook 01 supervises input evidence; notebook 03 supervises visual sources.


## Reproduce the benchmark inputs

Change `CADASTRAL_ROOT_ID` and run from the beginning. This notebook is independent of notebook 01 and prepares its own Catastro, LiDAR, and Roofer artifacts.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import laspy
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go

from urbanstock3d.processors.lidar import points_in_polygon
from urbanstock3d.providers.pnoa_lidar import footprint_polygons_utm
from urbanstock3d.providers.roofer import RooferClient
from urbanstock3d.reconstruction.execution import execute_reconstruction_plan
from urbanstock3d.reconstruction.execution import finalize_or_execute_fallback
from urbanstock3d.reconstruction.models import (
    GeometryProvenance,
    ReconstructionEvidence,
    ReconstructionRequest,
    ReconstructionResult,
)
from urbanstock3d.reconstruction.enums import BackendName, ReconstructionStatus
from urbanstock3d.reconstruction.backends import BackendRegistry, RooferBackend
from urbanstock3d.reconstruction.benchmark import (
    build_benchmark_entry,
    compare_reconstructions,
)
from urbanstock3d.reconstruction.planner import plan_reconstruction
from urbanstock3d.reconstruction.quality.lidar import assess_lidar_quality
from urbanstock3d.reconstruction.quality.lod_feasibility import (
    LodEvidenceAvailability,
    assess_lod_feasibility,
)
from urbanstock3d.reconstruction.quality.ransac import RansacParameters, detect_roof_planes
from urbanstock3d.reconstruction.quality.roof_complexity import assess_roof_complexity
from urbanstock3d.reconstruction.validation import (
    assess_cityjson_lidar_fit,
    assess_obj_lidar_fit,
    compute_cityjson_lidar_residuals,
    compute_obj_lidar_residuals,
    evaluate_reconstruction_quality,
    read_obj,
    select_roof_observations,
    validate_cityjsonseq,
    validate_obj,
)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RUN_BENCHMARK_PIPELINE = True
RUN_PLANNED_RECONSTRUCTION = False
RUN_QUALITY_FALLBACK = False
RUN_BATCH_BENCHMARK = True
# Change only this 14-character cadastral root to benchmark another building.
CADASTRAL_ROOT_ID = "4531917YJ2743B"
assert len(CADASTRAL_ROOT_ID) == 14, "Expected a 14-character cadastral root"
BUILDING_ID = f"ES.SDGC.BU.{CADASTRAL_ROOT_ID}"
BUILDING_PATH = PROJECT_ROOT / "outputs" / BUILDING_ID / "building.geojson"
CROP_PATH = PROJECT_ROOT / "outputs" / BUILDING_ID / "lidar_context_crop.laz"
ROOFER_ROOT = PROJECT_ROOT / "outputs" / BUILDING_ID / "roofer"
ROOFER_EXECUTABLE = PROJECT_ROOT / ".tools" / "roofer-v1.0.0" / "bin" / "roofer.exe"

In [ ]:
# Benchmark mode deliberately prepares Roofer and the shared evidence from a clean state.
if RUN_BENCHMARK_PIPELINE:
    scripts_directory = PROJECT_ROOT / "scripts"
    urbanstock_executable = Path(sys.executable).with_name(
        "urbanstock.exe" if os.name == "nt" else "urbanstock"
    )
    commands = [
        [
            str(urbanstock_executable),
            "resolve",
            "--refcat",
            CADASTRAL_ROOT_ID,
            "--output-dir",
            str(PROJECT_ROOT / "outputs"),
        ],
        [
            sys.executable,
            str(scripts_directory / "process_lidar_crop.py"),
            str(BUILDING_PATH),
            "--buffer-m",
            "25",
            "--save-crop",
            str(CROP_PATH),
            "--output",
            str(PROJECT_ROOT / "outputs" / BUILDING_ID / "lidar_crop_audit.json"),
        ],
        [
            sys.executable,
            str(scripts_directory / "prepare_roofer_input.py"),
            str(BUILDING_PATH),
            str(ROOFER_ROOT / "footprint_25830.geojson"),
        ],
        [
            sys.executable,
            str(scripts_directory / "run_roofer.py"),
            str(CROP_PATH),
            str(ROOFER_ROOT / "footprint_25830.geojson"),
            str(ROOFER_ROOT / "native_cityjson"),
            "--executable",
            str(ROOFER_EXECUTABLE),
            "--jobs",
            "1",
        ],
    ]
    for command in commands:
        print(f"Running: {' '.join(command)}")
        subprocess.run(command, cwd=PROJECT_ROOT, check=True)
else:
    print("Benchmark preparation skipped; existing artifacts will be used.")

## Benchmark coverage dashboard

This early dashboard evaluates artifacts that already exist; it does not launch missing reconstructions. It shows overall coverage before the detailed single-building analysis below. Missing inputs, missing models, and unsupported outputs remain visible because benchmark coverage is part of the evidence. Backend averages must not be interpreted until both methods cover a sufficiently varied common sample.


In [ ]:
batch_report_path = PROJECT_ROOT / "outputs" / "selected_buildings_reconstruction_benchmark.json"
if RUN_BATCH_BENCHMARK:
    subprocess.run(
        [sys.executable, str(PROJECT_ROOT / "scripts" / "benchmark_selected_buildings.py"),
         "--selection", str(PROJECT_ROOT / "data" / "selected_buildings.json"),
         "--outputs-root", str(PROJECT_ROOT / "outputs"),
         "--output", str(batch_report_path)],
        cwd=PROJECT_ROOT, check=True,
    )

batch_report = json.loads(batch_report_path.read_text(encoding="utf-8"))
coverage_rows = []
metric_rows = []
for candidate in batch_report["buildings"]:
    short_id = candidate["building_id"].split(".")[-1]
    entries = {entry["backend"]: entry for entry in candidate["entries"]}
    coverage_rows.append([short_id] + [
        ("PASS" if entries[backend]["accepted"] else "REJECT")
        if backend in entries else "NOT EVALUATED"
        for backend in ("roofer", "city3d")
    ])
    metric_rows.extend((short_id, entry["backend"], entry["point_surface_rmse_m"])
                       for entry in candidate["entries"])

fig, axes = plt.subplots(2, 1, figsize=(13, 7), constrained_layout=True)
axes[0].axis("off")
coverage_table = axes[0].table(
    cellText=coverage_rows, colLabels=["REFCAT", "Roofer", "City3D"],
    loc="center", cellLoc="center",
)
coverage_table.auto_set_font_size(False)
coverage_table.set_fontsize(9)
coverage_table.scale(1, 1.35)
axes[0].set_title("Benchmark coverage and acceptance")
backend_colors = {"roofer": "#d62728", "city3d": "#d97706"}
for backend in ("roofer", "city3d"):
    rows = [row for row in metric_rows if row[1] == backend]
    axes[1].scatter([row[0] for row in rows], [row[2] for row in rows],
                    label=backend, color=backend_colors[backend], s=65)
axes[1].axhline(1.0, color="#2ca02c", linestyle="--", label="RMSE gate (1 m)")
axes[1].set(title="Point-to-surface RMSE for evaluable artifacts",
            xlabel="Cadastral root", ylabel="RMSE (m)")
axes[1].tick_params(axis="x", rotation=30)
axes[1].legend()
plt.show()
print(json.dumps(batch_report["summaries"], indent=2))

## Load the common reconstruction evidence


In [ ]:
assert CROP_PATH.exists(), f"Missing LiDAR crop: {CROP_PATH}"
assert BUILDING_PATH.exists(), f"Missing building geometry: {BUILDING_PATH}"

cloud = laspy.read(CROP_PATH)
x = np.asarray(cloud.x)
y = np.asarray(cloud.y)
z = np.asarray(cloud.z)
classification = np.asarray(cloud.classification, dtype=np.uint8)
ground = classification == 2
assert ground.any(), "The context contains no classified ground points"
ground_z = float(np.median(z[ground]))
height = z - ground_z

building = json.loads(BUILDING_PATH.read_text(encoding="utf-8"))
footprint_polygons = footprint_polygons_utm(building["geometry"])
in_footprint = np.zeros(x.shape, dtype=bool)
for rings in footprint_polygons:
    in_footprint |= points_in_polygon(x, y, rings)

roof = in_footprint & (classification == 6)
roof_x = x[roof]
roof_y = y[roof]
roof_height = height[roof]
roof_points = np.column_stack((roof_x, roof_y, roof_height))
assert len(roof_points), "The footprint contains no classified roof points"

print(f"Benchmark building: {BUILDING_ID}")
print(f"Roof points: {len(roof_points):,}")
print(f"Ground reference: {ground_z:.2f} m")

## Select comparable roof observations

Class 6 means *building*, not necessarily visible roof. A shared local upper-envelope filter removes low facade or occluded returns before measuring either backend. Sparse neighborhoods are retained, and the audit counts below make the transformation explicit. This is evaluation preprocessing; it does not alter the reconstruction input.


In [ ]:
observation_mask, observation_report = select_roof_observations(roof_points)
print(json.dumps(observation_report.to_dict(), indent=2))

observation_figure = go.Figure()
for selected, label, color, opacity in (
    (False, "Rejected low building returns", "#d97706", 0.55),
    (True, "Selected roof observations", "#1f77b4", 0.85),
):
    candidate = roof_points[observation_mask == selected]
    observation_figure.add_trace(
        go.Scatter3d(
            x=candidate[:, 0], y=candidate[:, 1], z=candidate[:, 2],
            mode="markers", name=label, opacity=opacity,
            marker={"size": 2, "color": color},
        )
    )
observation_figure.update_layout(
    title="Shared LiDAR observations used by every backend metric", height=700,
    scene={"aspectmode": "data", "xaxis_title": "Easting (m)",
           "yaxis_title": "Northing (m)",
           "zaxis_title": "Height above local ground (m)"},
)
observation_figure.show()

## Production pre-routing evidence

Invoke the tested bounded RANSAC implementation before reconstruction. The report measures plane support, residuals, slopes and dominant normal groups; it does not generate roof polygons.


In [ ]:
core_ransac_parameters = RansacParameters()
core_plane_evidence = detect_roof_planes(roof_points, parameters=core_ransac_parameters)
print(json.dumps(core_plane_evidence.to_dict(), indent=2))

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
plane_support = [plane.support_ratio for plane in core_plane_evidence.planes]
plane_slopes = [plane.slope_deg for plane in core_plane_evidence.planes]
axes[0].bar(range(1, len(plane_support) + 1), plane_support, color="#1f77b4")
axes[0].set(title="Support per preliminary plane", xlabel="Plane", ylabel="Fraction of sampled roof points")
axes[1].bar(range(1, len(plane_slopes) + 1), plane_slopes, color="#d62728")
axes[1].set(title="Plane slopes", xlabel="Plane", ylabel="Slope (degrees)")
plt.show()


## Route one production reconstruction

The planner combines measured quality, complexity, LoD feasibility and the actually available native Roofer adapter. It selects one primary execution and at most one lower-LoD fallback; this cell does not run reconstruction.


In [ ]:
lidar_quality, _ = assess_lidar_quality(CROP_PATH, footprint_polygons)
building_parts_path = PROJECT_ROOT / "outputs" / BUILDING_ID / "building_parts.geojson"
building_part_count = len(json.loads(building_parts_path.read_text(encoding="utf-8"))["features"])
roof_complexity = assess_roof_complexity(
    footprint_polygons, core_plane_evidence, building_part_count=building_part_count
)
lod_feasibility = assess_lod_feasibility(
    LodEvidenceAvailability(True, bool(ground.any()), bool(roof.any())),
    lidar_quality,
    roof_complexity,
)
backend_registry = BackendRegistry()
backend_registry.register(
    RooferBackend(RooferClient(str(ROOFER_EXECUTABLE)), ROOFER_ROOT / "planned_runs")
)
reconstruction_plan = plan_reconstruction(
    ReconstructionRequest(),
    lidar_quality,
    roof_complexity,
    lod_feasibility,
    backend_registry,
)
routing_summary = {
    "requested_lod": reconstruction_plan.requested_lod.value,
    "target_lod": reconstruction_plan.target_lod.value if reconstruction_plan.target_lod else None,
    "backend": reconstruction_plan.selected_backend.value if reconstruction_plan.selected_backend else None,
    "profile": reconstruction_plan.selected_profile,
    "fallback_lod": reconstruction_plan.fallback_lod.value if reconstruction_plan.fallback_lod else None,
    "reasons": reconstruction_plan.reasons,
}
print(json.dumps(routing_summary, indent=2))


## Execute the selected production plan (optional)

Set `RUN_PLANNED_RECONSTRUCTION=True` only when you want to launch the selected backend. The executor runs exactly one primary backend. Fallback execution remains disabled until independent output QC exists.


In [ ]:
if RUN_PLANNED_RECONSTRUCTION:
    reconstruction_evidence = ReconstructionEvidence(
        building_id=BUILDING_ID,
        footprint=ROOFER_ROOT / "footprint_25830.geojson",
        lidar_points=CROP_PATH,
    )
    reconstruction_result = execute_reconstruction_plan(
        reconstruction_plan, reconstruction_evidence, backend_registry
    )
    execution_summary = {
        "status": reconstruction_result.status.value,
        "requested_lod": reconstruction_result.requested_lod.value,
        "targeted_lod": reconstruction_result.targeted_lod.value if reconstruction_result.targeted_lod else None,
        "delivered_lod": reconstruction_result.delivered_lod.value if reconstruction_result.delivered_lod else None,
        "backend": reconstruction_result.backend.value if reconstruction_result.backend else None,
        "model_path": str(reconstruction_result.model_path) if reconstruction_result.model_path else None,
        "reasons": reconstruction_result.reasons,
    }
    print(json.dumps(execution_summary, indent=2))
else:
    print("Planned execution skipped. Existing benchmark output is visualized below.")


In [ ]:
roofer_root = PROJECT_ROOT / "outputs" / BUILDING_ID / "roofer"
native_output_directory = roofer_root / "native_cityjson"
roofer_output_directory = (
    native_output_directory if native_output_directory.exists() else roofer_root / "cityjson"
)
roofer_files = sorted(roofer_output_directory.glob("*.city.jsonl"))
assert roofer_files, f"No Roofer CityJSONSeq found in {roofer_output_directory}"

roofer_records = [
    json.loads(line)
    for line in roofer_files[0].read_text(encoding="utf-8").splitlines()
    if line.strip()
]
roofer_metadata = next(record for record in roofer_records if record["type"] == "CityJSON")
roofer_features = [record for record in roofer_records if record["type"] == "CityJSONFeature"]
roofer_reconstructions = []
for feature in roofer_features:
    building_object = feature["CityObjects"].get(BUILDING_ID)
    if building_object is None:
        continue
    for city_object in feature["CityObjects"].values():
        if city_object["type"] != "BuildingPart":
            continue
        for geometry in city_object.get("geometry", []):
            if geometry.get("lod") == "2.2":
                roofer_reconstructions.append(
                    {"feature": feature, "building": building_object, "lod22": geometry}
                )

assert roofer_reconstructions, "Roofer produced no BuildingPart with LoD 2.2"
representative_reconstruction = min(
    roofer_reconstructions,
    key=lambda item: item["building"]["attributes"]["rf_nodata_frac"],
)
roofer_building = representative_reconstruction["building"]
roofer_attributes = roofer_building["attributes"]
print(f"LoD2.2 reconstructed components: {len(roofer_reconstructions)} / {len(roofer_features)}")

comparison = {
    "process_succeeded": roofer_attributes["rf_success"],
    "extrusion_mode": roofer_attributes["rf_extrusion_mode"],
    "roof_type": roofer_attributes["rf_roof_type"],
    "detected_planes": roofer_attributes["rf_roof_planes"],
    "ridge_lines": roofer_attributes["rf_ridgelines"],
    "point_density_m2": roofer_attributes["rf_pt_density"],
    "nodata_fraction": roofer_attributes["rf_nodata_frac"],
    "lod22_rmse_m": roofer_attributes["rf_rmse_lod22"],
}
for key, value in comparison.items():
    print(f"{key}: {value}")


print("Native attributes are diagnostics only; acceptance is decided by common core QC below.")


### Common structural output quality

This is the first backend-independent benchmark gate: valid semantic solid, watertight boundary, plausible height, and coarse footprint agreement. Passing it does not yet mean the roof fits the LiDAR; point-to-surface accuracy is the next validation block.


In [ ]:
structural_quality = validate_cityjsonseq(roofer_files[0], footprint_polygons)
lidar_fit = assess_cityjson_lidar_fit(roofer_files[0], CROP_PATH, footprint_polygons)
final_quality = evaluate_reconstruction_quality(structural_quality, lidar_fit)
print(json.dumps(structural_quality.to_dict(), indent=2))
print(json.dumps(lidar_fit.to_dict(), indent=2))
print(json.dumps(final_quality.to_dict(), indent=2))
quality_labels = ["Geometry", "Watertight", "Plausibility", "Footprint bbox"]
quality_values = [
    float(structural_quality.geometry_valid),
    float(structural_quality.watertight),
    float(structural_quality.plausibility_pass),
    structural_quality.footprint_bbox_iou,
]
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(quality_labels, quality_values, color=["#2ca02c" if value >= 0.8 else "#d62728" for value in quality_values])
ax.bar_label(bars, fmt="%.3f")
ax.set(ylim=(0, 1.12), ylabel="Check value", title="Backend-independent structural QC")
plt.show()
print(f"Structural gate accepted: {structural_quality.accepted}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
distance_names = ["Median", "RMSE", "P95"]
distance_values = [lidar_fit.distance_median_m, lidar_fit.distance_rmse_m, lidar_fit.distance_p95_m]
axes[0].bar(distance_names, distance_values, color="#d62728")
axes[0].set(ylabel="3D distance (m)", title="Observed roof to reconstructed surface")
support_names = ["<=0.2 m", "<=0.5 m", "<=1.0 m"]
support_values = [lidar_fit.within_020m_ratio, lidar_fit.within_050m_ratio, lidar_fit.within_100m_ratio]
axes[1].bar(support_names, support_values, color="#1f77b4")
axes[1].set(ylim=(0, 1), ylabel="Roof-point fraction", title="Metric support by tolerance")
plt.show()
print(f"FINAL DECISION: {'ACCEPTED' if final_quality.accepted else 'REJECTED'} ({final_quality.confidence_class.value})")
for failure in final_quality.failures:
    print(f"- {failure}")
roofer_benchmark_entry = build_benchmark_entry(
    BackendName.ROOFER, structural_quality, lidar_fit, final_quality
)
benchmark_report = compare_reconstructions((roofer_benchmark_entry,))
benchmark_columns = ["Backend", "LoD", "Accepted", "RMSE m", "P95 m", "<=0.5 m", "Faces"]
benchmark_rows = [[
    entry.backend.value, entry.lod, str(entry.accepted),
    f"{entry.point_surface_rmse_m:.2f}", f"{entry.point_surface_p95_m:.2f}",
    f"{entry.within_050m_ratio:.1%}", str(entry.face_count),
] for entry in benchmark_report.entries]
fig, ax = plt.subplots(figsize=(12, 2.2))
ax.axis("off")
table = ax.table(cellText=benchmark_rows, colLabels=benchmark_columns, loc="center", cellLoc="center")
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.5)
ax.set_title("Common reconstruction benchmark - implemented backends only")
plt.show()
print(f"Accepted ranking: {[backend.value for backend in benchmark_report.accepted_ranking] or 'none'}")
planned_fallback = reconstruction_plan.fallback_lod if not final_quality.accepted else None
print(f"Planned quality fallback: {planned_fallback or 'none'}")
if RUN_QUALITY_FALLBACK and not final_quality.accepted:
    primary_candidate = ReconstructionResult(
        status=ReconstructionStatus.SUCCESS,
        requested_lod=reconstruction_plan.requested_lod,
        targeted_lod=reconstruction_plan.target_lod,
        delivered_lod=reconstruction_plan.target_lod,
        backend=reconstruction_plan.selected_backend,
        model_path=roofer_files[0],
        provenance=GeometryProvenance(lidar_observed=True),
    )
    fallback_evidence = ReconstructionEvidence(
        BUILDING_ID, ROOFER_ROOT / "footprint_25830.geojson", lidar_points=CROP_PATH
    )
    fallback_result = finalize_or_execute_fallback(
        reconstruction_plan, fallback_evidence, backend_registry, primary_candidate, final_quality
    )
    print(f"Fallback status: {fallback_result.status.value}; target: {fallback_result.targeted_lod}")


### Spatial residual diagnostics

Aggregate metrics cannot reveal whether an error is isolated, follows a roof edge, or affects an entire component. This view colors each selected LiDAR observation by its exact 3D distance to Roofer. The scale is capped at 3 m so a few extreme outliers do not hide the spatial pattern; hover text retains the uncapped value.


In [ ]:
roofer_residuals = compute_cityjson_lidar_residuals(
    roofer_files[0], CROP_PATH, footprint_polygons
)
residual_points = roofer_residuals.points
residual_values = roofer_residuals.distances_m
residual_figure = go.Figure(
    go.Scatter3d(
        x=residual_points[:, 0], y=residual_points[:, 1],
        z=residual_points[:, 2] - ground_z, mode="markers",
        customdata=residual_values,
        hovertemplate="Residual: %{customdata:.3f} m<extra></extra>",
        marker={"size": 3, "color": np.minimum(residual_values, 3.0),
                "colorscale": "Turbo", "cmin": 0, "cmax": 3,
                "colorbar": {"title": "Distance (m)"}},
    )
)
residual_figure.update_layout(
    title="Roofer spatial point-to-surface residuals", height=720,
    scene={"aspectmode": "data", "xaxis_title": "Easting (m)",
           "yaxis_title": "Northing (m)",
           "zaxis_title": "Height above local ground (m)"},
)
residual_figure.show()
worst = np.argsort(residual_values)[-10:][::-1]
print("Ten largest residuals (metres):", np.round(residual_values[worst], 3).tolist())

In [ ]:
scale = np.asarray(roofer_metadata["transform"]["scale"])
translation = np.asarray(roofer_metadata["transform"]["translate"])
surface_colors = {"GroundSurface": "#8c564b", "WallSurface": "#7f7f7f", "RoofSurface": "#d62728"}
roofer_figure = go.Figure()
legend_types = set()

for reconstruction in roofer_reconstructions:
    feature = reconstruction["feature"]
    lod22 = reconstruction["lod22"]
    roofer_vertices = np.asarray(feature["vertices"]) * scale + translation
    semantic_surfaces = lod22["semantics"]["surfaces"]
    semantic_values = lod22["semantics"]["values"][0]
    for boundary, semantic_index in zip(lod22["boundaries"][0], semantic_values, strict=True):
        surface_type = semantic_surfaces[semantic_index]["type"]
        color = surface_colors[surface_type]
        for ring in boundary:
            closed_ring = [*ring, ring[0]]
            coordinates = roofer_vertices[closed_ring]
            roofer_figure.add_trace(
                go.Scatter3d(
                    x=coordinates[:, 0],
                    y=coordinates[:, 1],
                    z=coordinates[:, 2] - ground_z,
                    mode="lines",
                    name=surface_type,
                    legendgroup=surface_type,
                    showlegend=surface_type not in legend_types,
                    line={"color": color, "width": 5},
                )
            )
            legend_types.add(surface_type)

roofer_figure.update_layout(
    title="Roofer LoD2.2 semantic geometry",
    scene={
        "xaxis_title": "Easting (m)",
        "yaxis_title": "Northing (m)",
        "zaxis_title": "Height above local ground (m)",
        "aspectmode": "data",
    },
    height=750,
)
roofer_figure.show()

## 5. Evaluate the City3D OBJ candidate

City3D is evaluated with the same structural and LiDAR-fit thresholds as Roofer. The wrapper uses the median of classified ground returns instead of the raw point-cloud minimum. Because OBJ has no semantic labels, roof faces are inferred from orientation and elevation and this limitation is retained in the report.

In [ ]:
city3d_model = PROJECT_ROOT / "outputs" / BUILDING_ID / "city3d" / "building.obj"
if not city3d_model.exists():
    city3d_model = PROJECT_ROOT / "outputs" / BUILDING_ID / "city3d" / "building_python.obj"

if city3d_model.exists():
    city3d_geometry = validate_obj(city3d_model, footprint_polygons)
    city3d_lidar_fit = assess_obj_lidar_fit(city3d_model, CROP_PATH, footprint_polygons)
    city3d_residuals = compute_obj_lidar_residuals(city3d_model, CROP_PATH, footprint_polygons)
    city3d_quality = evaluate_reconstruction_quality(city3d_geometry, city3d_lidar_fit)
    print(json.dumps(city3d_geometry.to_dict(), indent=2))
    roofer_entry = build_benchmark_entry(BackendName.ROOFER, structural_quality, lidar_fit, final_quality)
    city3d_entry = build_benchmark_entry(BackendName.CITY3D, city3d_geometry, city3d_lidar_fit, city3d_quality)
    comparison = compare_reconstructions((roofer_entry, city3d_entry))
    print(json.dumps(comparison.to_dict(), indent=2))

    city3d_vertices, city3d_faces = read_obj(city3d_model)
    city3d_triangles = [
        (face[0], face[index], face[index + 1])
        for face in city3d_faces
        for index in range(1, len(face) - 1)
    ]
    city3d_figure = go.Figure(
        go.Mesh3d(
            x=city3d_vertices[:, 0], y=city3d_vertices[:, 1],
            z=city3d_vertices[:, 2] - city3d_vertices[:, 2].min(),
            i=[triangle[0] for triangle in city3d_triangles],
            j=[triangle[1] for triangle in city3d_triangles],
            k=[triangle[2] for triangle in city3d_triangles],
            color="#d97706", opacity=0.82, flatshading=True, name="City3D",
        )
    )
    city3d_figure.add_trace(
        go.Scatter3d(
            x=city3d_residuals.points[:, 0], y=city3d_residuals.points[:, 1],
            z=city3d_residuals.points[:, 2] - ground_z, mode="markers",
            customdata=city3d_residuals.distances_m,
            hovertemplate="Residual: %{customdata:.3f} m<extra></extra>",
            marker={"size": 2, "color": np.minimum(city3d_residuals.distances_m, 3.0),
                    "colorscale": "Turbo", "cmin": 0, "cmax": 3,
                    "colorbar": {"title": "Distance (m)"}},
            name="LiDAR residuals",
        )
    )
    city3d_figure.update_layout(
        title="City3D reconstructed OBJ", height=750,
        scene={"aspectmode": "data", "xaxis_title": "Easting (m)",
               "yaxis_title": "Northing (m)", "zaxis_title": "Relative height (m)"},
    )
    city3d_figure.show()
else:
    print(f"City3D model not found: {city3d_model}. Run scripts/run_city3d.py first.")

## Human review checklist

- Does filtering discard actual lower roof levels?
- Are core planes supported by the visible cloud?
- Does backend/LoD selection match core evidence?
- Is the model aligned with Catastro?
- Are large residuals isolated, along edges or across whole components?
- Is geometry watertight and plausible?
- Do missing artifacts prevent fair comparison?
- Should failed LoD2.2 fall back to lower LoD or abstain?
